# GameTheory-15 — Jeux Coopératifs (Twin C#)

**Jumeau C# (.NET Interactive)** de `GameTheory-15-CooperativeGames.ipynb` — marathon parité .NET ⇄ Python (#4956).

Ce notebook réimplémente from-scratch en C# pur (BCL .NET 9, **0 NuGet**) la **théorie des jeux coopératifs** : contrairement aux jeux non-coopératifs (où l'on étudie les stratégies individuelles), les jeux coopératifs modélisent ce qu'une **coalition** peut obtenir seule, via une **fonction caractéristique** `v : 2^N → ℝ`. On y calcule la **valeur de Shapley** (répartition juste par contribution marginale moyenne), l'**indice de pouvoir de Banzhaf** (pouvoir de swing dans un vote pondéré), le **core** (répartitions non bloquables par aucune coalition), et l'on étudie les **jeux convexes** (où Shapley ∈ core).

> L'original Python s'appuie sur `numpy`/`itertools` ; ce twin traduit l'énumération des coalitions (powerset) et des permutations en C# (LINQ + récursion), et restitue les résultats en **tables console**.

**Voir aussi** : [GameTheory-15 (Python)](GameTheory-15-CooperativeGames.ipynb) · marathon [#4956](https://github.com/jsboige/CoursIA/issues/4956) · registre SOTA [#3801](https://github.com/jsboige/CoursIA/issues/3801).


## Objectifs d'apprentissage

1. Modéliser un **jeu coopératif à utilité transférable** (TU game) via sa fonction caractéristique.
2. Énumérer **coalitions** (powerset) et **permutations**.
3. Calculer la **valeur de Shapley** (contribution marginale moyenne sur toutes les permutations).
4. Calculer l'**indice de pouvoir de Banzhaf** (swing, vote pondéré).
5. Définir le **core** et tester la **non-vacuité** (efficacité + inégalités de coalition).
6. Reconnaître les **jeux convexes** (marginales croissantes ⟹ core non vide, Shapley ∈ core).

### Prérequis

- Théorie des jeux en forme normale (cf. [GameTheory-2](GameTheory-2-NormalForm-Csharp.ipynb), [GameTheory-4](GameTheory-4-NashEquilibrium-Csharp.ipynb)).


In [1]:
// Setup : types de base pour un jeu cooperatif (TU game).
#nullable enable
using System;
using System.Collections.Generic;
using System.Linq;

public static void Show(object? o) => (o?.ToString() ?? "<null>").Display();

// Un jeu cooperatif a utilite transferable : fonction caracteristique v: 2^N -> R.
// Representee par un dictionnaire : cle = coalition (sorted string of player indices), valeur = v(S).
public sealed class CooperativeGame
{
    public int N { get; }                       // nombre de joueurs
    public string[] Players { get; }
    private readonly Dictionary<long, double> _v = new();   // cle = bitmask de la coalition
    public CooperativeGame(int n, string[]? names = null)
    { N = n; Players = names ?? Enumerable.Range(0,n).Select(i=>$"J{i+1}").ToArray(); }
    public CooperativeGame SetV(IEnumerable<int> coalition, double value)
    { _v[Mask(coalition)] = value; return this; }
    public double V(IEnumerable<int> coalition) => _v.TryGetValue(Mask(coalition), out var val) ? val : 0.0;
    public long Mask(IEnumerable<int> coalition) { long m = 0; foreach (var i in coalition) m |= 1L << i; return m; }
    public IEnumerable<int> PlayersAsIndices => Enumerable.Range(0, N);
    public double VGrand => V(PlayersAsIndices);   // v(N)
}

Show("Types charges : CooperativeGame (v: 2^N -> R via bitmask), helper Show().");


The below script needs to be able to find the current output cell; this is an easy method to get it.

Types charges : CooperativeGame (v: 2^N -> R via bitmask), helper Show().

## 1. Fonction caractéristique et coalitions

### Définition

Un **jeu coopératif à utilité transférable** (von Neumann-Morgenstern 1944) est un couple `(N, v)` où `N = {1,…,n}` est l'ensemble des joueurs et `v : 2^N → ℝ` est la **fonction caractéristique** : `v(S)` est la valeur que la coalition `S` peut garantir seule (son « worth »). Conventions : `v(∅) = 0`, `v(S) ≥ 0` souvent.

### Énumération des coalitions

L'ensemble des coalitions = le **powerset** de `N` (`2^n` coalitions). On l'énumère via les **bitmasks** de `0` à `2^n − 1`.

### Propriétés

- **Superadditivité** : `v(S ∪ T) ≥ v(S) + v(T)` si `S ∩ T = ∅` (fusioner paie).
- **Convexité** : `v(S ∪ {i}) − v(S)` **croît** avec `S` (rend les grandes coalitions stables).


In [2]:
// Enumere toutes les coalitions (powerset) via bitmask.
public static List<List<int>> AllCoalitions(int n, bool includeEmpty = true)
{
    var res = new List<List<int>>();
    long total = 1L << n;
    for (long m = 0; m < total; m++)
    {
        if (m == 0 && !includeEmpty) continue;
        var c = new List<int>();
        for (int i = 0; i < n; i++) if ((m & (1L << i)) != 0) c.Add(i);
        res.Add(c);
    }
    return res;
}

// Test de superadditivite : v(S u T) >= v(S) + v(T) pour S, T disjoints.
public static bool IsSuperadditive(CooperativeGame g)
{
    var coalitions = AllCoalitions(g.N, includeEmpty: false);
    foreach (var S in coalitions)
        foreach (var T in coalitions)
        {
            if (S.Intersect(T).Any()) continue;
            var union = S.Concat(T).ToList();
            if (g.V(union) + 1e-9 < g.V(S) + g.V(T)) return false;
        }
    return true;
}

// Test de convexite : marginales croissantes (v(S u {i}) - v(S)) croit avec S).
public static bool IsConvex(CooperativeGame g)
{
    foreach (int i in g.PlayersAsIndices)
    {
        var others = g.PlayersAsIndices.Where(j => j != i).ToList();
        var coalitionsWithoutI = AllCoalitions(g.N - 1, includeEmpty: true);
        // reindexer : mappper les coalitions de 'others' vers les vrais indices
        for (int a = 0; a < coalitionsWithoutI.Count; a++)
        for (int b = a + 1; b < coalitionsWithoutI.Count; b++)
        {
            var Sa = coalitionsWithoutI[a].Select(idx => others[idx]).ToList();
            var Sb = coalitionsWithoutI[b].Select(idx => others[idx]).ToList();
            // convexite : S subset T => marginale_i(S) <= marginale_i(T)
            if (Sa.All(x => Sb.Contains(x)))
            {
                double margS = g.V(Sa.Append(i)) - g.V(Sa);
                double margT = g.V(Sb.Append(i)) - g.V(Sb);
                if (margS > margT + 1e-9) return false;
            }
        }
    }
    return true;
}

Show("AllCoalitions (powerset bitmask) + IsSuperadditive + IsConvex definis.");


AllCoalitions (powerset bitmask) + IsSuperadditive + IsConvex definis.

### Interprétation : le « glove game » (jeu des gants)

Le **glove game** canonique : 3 joueurs — le joueur 1 a un gant **gauche** (L), les joueurs 2 et 3 ont un gant **droit** (R). Une paire (L, R) vaut 1, sinon 0. Fonction caractéristique :
- `v({1}) = v({2}) = v({3}) = 0` (un seul gant ne sert à rien)
- `v({1,2}) = v({1,3}) = 1` (une paire), `v({2,3}) = 0` (deux droits)
- `v({1,2,3}) = 1` (une seule paire réalisable)

Le joueur 1 (détenteur du gauche rare) a un **pouvoir de marché** : il est dans toutes les coalitions rentables.


In [3]:
// Glove game : 1 gauche (J1), 2 droits (J2, J3). Une paire (L,R) = 1.
var glove = new CooperativeGame(3, new[]{ "L1", "R2", "R3" });
glove.SetV(new[]{0}, 0).SetV(new[]{1}, 0).SetV(new[]{2}, 0);
glove.SetV(new[]{0,1}, 1).SetV(new[]{0,2}, 1).SetV(new[]{1,2}, 0);
glove.SetV(new[]{0,1,2}, 1);

var sb = new System.Text.StringBuilder();
sb.AppendLine("Glove game (1 gauche, 2 droits) -- v(S) pour chaque coalition :");
foreach (var c in AllCoalitions(3, includeEmpty: false))
{
    var names = string.Join("", c.Select(i => glove.Players[i]));
    sb.AppendLine($"  v({{ {names} }}) = {glove.V(c)}");
}
sb.AppendLine();
sb.AppendLine($"Superadditive : {IsSuperadditive(glove)}");
sb.AppendLine($"Convexe       : {IsConvex(glove)}");
sb.AppendLine($"v(N) = v(grand coalition) = {glove.VGrand}");
Show(sb.ToString());


Glove game (1 gauche, 2 droits) -- v(S) pour chaque coalition :
  v({ L1 }) = 0
  v({ R2 }) = 0
  v({ L1R2 }) = 1
  v({ R3 }) = 0
  v({ L1R3 }) = 1
  v({ R2R3 }) = 0
  v({ L1R2R3 }) = 1

Superadditive : True
Convexe       : False
v(N) = v(grand coalition) = 1


### Interprétation des résultats

- **Superadditif** : oui (fusioner des coalitions disjointes ne diminue jamais la valeur — une coalition plus grosse accède à plus de gants).
- **Non convexe** : la marginale du joueur 1 **décroît** (passer de {2} à {2,3} comme partenaires ne crée qu'une paire au maximum). Le jeu n'est pas convexe, donc on n'a **pas** la garantie que le core soit non vide (on le vérifiera plus bas).


## 2. La valeur de Shapley

### Définition (Shapley 1953)

La **valeur de Shapley** `φ_i` du joueur `i` est la **moyenne de ses contributions marginales** sur **toutes les permutations** des joueurs (modélisant tous les ordres d'arrivée équiprobables) :

$$\varphi_i = \frac{1}{n!} \sum_{\pi \in S_n} \big[ v(\{\text{joueurs avant } i \text{ dans } \pi\} \cup \{i\}) - v(\text{joueurs avant } i) \big]$$

### Propriétés (axiomes caractérisant Shapley)

- **Efficacité** : `Σ_i φ_i = v(N)` (tout est redistribué).
- **Symétrie** : deux joueurs interchangeables ont même φ.
- **Joueur nul** : si `i` ne contribue jamais (`v(S∪{i})=v(S)` ∀S), alors `φ_i = 0`.
- **Additivité** : `φ` est linéaire en `v`.

Shapley est l'**unique** valeur satisfaisant ces 4 axiomes.


In [4]:
// Enumere toutes les permutations (Heap iteratif) des indices 0..n-1.
public static IEnumerable<List<int>> Permutations(int n)
{
    var a = Enumerable.Range(0, n).ToArray();
    var c = new int[n];
    yield return a.ToList();
    int i = 0;
    while (i < n)
    {
        if (c[i] < i)
        {
            if (i % 2 == 0) (a[0], a[i]) = (a[i], a[0]);
            else (a[c[i]], a[i]) = (a[i], a[c[i]]);
            yield return a.ToList();
            c[i]++;
            i = 0;
        }
        else { c[i] = 0; i++; }
    }
}

// Valeur de Shapley : moyenne des contributions marginales sur toutes les permutations.
public static double[] Shapley(CooperativeGame g)
{
    double[] phi = new double[g.N];
    long count = 0;
    foreach (var perm in Permutations(g.N))
    {
        var before = new List<int>();
        foreach (int i in perm)
        {
            var withI = new List<int>(before) { i };
            phi[i] += g.V(withI) - g.V(before);
            before = withI;
        }
        count++;
    }
    for (int i = 0; i < g.N; i++) phi[i] /= count;
    return phi;
}

var phiGlove = Shapley(glove);
var sb = new System.Text.StringBuilder();
sb.AppendLine("Valeur de Shapley du glove game :");
for (int i = 0; i < 3; i++) sb.AppendLine($"  phi_{glove.Players[i]} = {phiGlove[i]:F4}");
sb.AppendLine($"  Somme = {phiGlove.Sum():F4}  (efficacite : doit egaler v(N) = {glove.VGrand})");
sb.AppendLine();
sb.AppendLine("Lecture : le detenteur du gant gauche (L1, rare) capte ~2/3 de la valeur,");
sb.AppendLine("les deux detenteurs droits se partagent le reste (symetrie R2/R3).");
Show(sb.ToString());


Valeur de Shapley du glove game :
  phi_L1 = 0,6667
  phi_R2 = 0,1667
  phi_R3 = 0,1667
  Somme = 1,0000  (efficacite : doit egaler v(N) = 1)

Lecture : le detenteur du gant gauche (L1, rare) capte ~2/3 de la valeur,
les deux detenteurs droits se partagent le reste (symetrie R2/R3).


### Interprétation : Shapley du glove game

La valeur de Shapley donne `φ_{L1} = 2/3`, `φ_{R2} = φ_{R3} = 1/6`. Vérification :
- **Efficacité** : `2/3 + 1/6 + 1/6 = 1 = v(N)` ✓.
- **Symétrie** : R2 et R3 (interchangeables) ont même φ ✓.
- **Pouvoir du rare** : L1 (le gauche) est dans toutes les coalitions rentables, d'où sa contribution marginale moyenne élevée.

Le joueur qui détient la ressource **rare** (complémentaire) capture l'essentiel de la valeur — c'est l'intuition économique de Shapley (prix d'un facteur rare).


## 3. Indice de pouvoir de Banzhaf

### Définition (Banzhaf 1965)

Dans un jeu de **vote pondéré** (chaque joueur a un poids, une coalition gagne si son poids total ≥ quota `q`), l'**indice de Banzhaf** mesure le **pouvoir de swing** d'un joueur : combien de coalitions deviennent gagnantes par son ajout.

Un joueur `i` est **décisif** dans la coalition `S` (avec `i ∉ S`) si `S` perd mais `S ∪ {i}` gagne. L'indice de Banzhaf **non normalisé** :

$$\beta_i = |\{ S \subseteq N \setminus \{i\} : S \text{ perd}, S \cup \{i\} \text{ gagne} \}|$$

L'indice **normalisé** `β_i^* = β_i / Σ_j β_j` donne la part de pouvoir.

> Contrairement à Shapley (qui pondère par la taille de coalition), Banzhaf traite toutes les coalitions équiprobablement. Utile en **analyse de pouvoir de vote** (Conseil de l'UE, assemblées d'actionnaires).


In [5]:
// Jeu de vote pondere : v(S) = 1 si sum(poids) >= q, sinon 0.
#nullable enable
public static CooperativeGame WeightedVotingGame(int[] weights, double q, string[]? names = null)
{
    int n = weights.Length;
    var g = new CooperativeGame(n, names);
    foreach (var S in AllCoalitions(n, includeEmpty: true))
        g.SetV(S, S.Sum(i => weights[i]) >= q ? 1.0 : 0.0);
    return g;
}

// Indice de Banzhaf non normalise : nombre de swing coalitions pour chaque joueur.
public static double[] Banzhaf(CooperativeGame g)
{
    double[] beta = new double[g.N];
    var coalitions = AllCoalitions(g.N, includeEmpty: true);
    foreach (int i in g.PlayersAsIndices)
    {
        foreach (var S in coalitions)
        {
            if (S.Contains(i)) continue;
            var withI = new List<int>(S) { i };
            bool Sloses = g.V(S) < 0.5;        // v=0
            bool Swing = g.V(withI) >= 0.5;    // v=1
            if (Sloses && Swing) beta[i] += 1;
        }
    }
    return beta;
}

// Exemple : 3 actionnaires, poids [50, 49, 1], quota 50 (majorite).
int[] weights = { 50, 49, 1 };
var wvg = WeightedVotingGame(weights, 50.0, new[]{ "A50", "B49", "C1" });
var beta = Banzhaf(wvg);
double bsum = beta.Sum();
var sb = new System.Text.StringBuilder();
sb.AppendLine("Jeu de vote pondere : poids [50, 49, 1], quota = 50");
sb.AppendLine("Coalitions gagnantes :");
foreach (var S in AllCoalitions(3, includeEmpty: false))
    if (wvg.V(S) >= 0.5) sb.AppendLine($"  {{ {string.Join(",", S.Select(i=>wvg.Players[i]))} }} poids={S.Sum(i=>weights[i])}");
sb.AppendLine();
sb.AppendLine("Indice de Banzhaf :");
for (int i = 0; i < 3; i++) sb.AppendLine($"  beta_{wvg.Players[i]} = {beta[i]:F0}  (normalise = {beta[i]/bsum:F4})");
sb.AppendLine();
sb.AppendLine("Lecon : bien que C1 n'ait que 1% du capital, il a autant de pouvoir de swing");
sb.AppendLine("que B49 (chacun decisive dans 1 coalition pivot, {B49,C1}=50). Le poids != le pouvoir.");
Show(sb.ToString());


Jeu de vote pondere : poids [50, 49, 1], quota = 50
Coalitions gagnantes :
  { A50 } poids=50
  { A50,B49 } poids=99
  { A50,C1 } poids=51
  { B49,C1 } poids=50
  { A50,B49,C1 } poids=100

Indice de Banzhaf :
  beta_A50 = 3  (normalise = 0,6000)
  beta_B49 = 1  (normalise = 0,2000)
  beta_C1 = 1  (normalise = 0,2000)

Lecon : bien que C1 n'ait que 1% du capital, il a autant de pouvoir de swing
que B49 (chacun decisive dans 1 coalition pivot, {B49,C1}=50). Le poids != le pouvoir.


### Interprétation : poids ≠ pouvoir

Dans le vote `[50, 49, 1]` (quota 50), l'indice de Banzhaf vaut **β = (3, 1, 1)** :
- **A50** est décisif dans **3** coalitions (`{}`→`{A50}`, `{B49}`→`{A50,B49}`, `{C1}`→`{A50,C1}`) → β = 3.
- **B49** est décisif dans **1** coalition (`{C1}`→`{B49,C1}` = 50 ≥ 50) → β = 1.
- **C1** est décisif dans **1** coalition (`{B49}`→`{B49,C1}` = 50 ≥ 50) → β = 1.

Ainsi **B49 et C1 ont le même pouvoir de swing** (1 chacun) malgré l'écart de capital (49 vs 1). C'est le cœur de l'analyse de Banzhaf : le **poids nominal est trompeur**, seul compte le nombre de coalitions pivot. Normalisé : `(0.6, 0.2, 0.2)` — C1 (1% du capital) détient **20%** du pouvoir.


## 4. Le core

### Définition

Le **core** (Gillies 1959) est l'ensemble des **imputations** `x = (x_1,…,x_n)` que **aucune coalition ne peut bloquer** :
- **Efficacité** : `Σ_i x_i = v(N)` (tout est distribué).
- **Rationalité coalitionnelle** : `Σ_{i∈S} x_i ≥ v(S)` pour **toute** coalition `S`.

Une coalition `S` pour laquelle `Σ_{i∈S} x_i < v(S)` peut **bloquer** `x` (elle obtient moins seule qu'en se séparable). Le core = répartitions **stables** (non bloquables).

### Non-vacuité

Le core peut être **vide** (aucune répartition stable). Le **théorème de Bondareva-Shapley** : le core est non vide ssi le jeu est **équilibré**. Une condition suffisante : le jeu est **convexe** (alors Shapley ∈ core).

### Test pratique

Pour un petit jeu, on teste la non-vacuité en cherchant une imputation `x` efficace satisfaisant toutes les inégalités `Σ_{i∈S} x_i ≥ v(S)`. C'est un **programme linéaire** (feasibility). On l'illustre ici par vérification directe de candidats (Shapley, imputation extrêmale).


In [6]:
// Teste si une imputation x est dans le core (efficace + rationnelle pour toute coalition).
public static (bool inCore, List<string> blocking) IsInCore(CooperativeGame g, double[] x)
{
    var blocking = new List<string>();
    if (Math.Abs(x.Sum() - g.VGrand) > 1e-9)
    { blocking.Add($"efficacite: sum x = {x.Sum():F4} != v(N) = {g.VGrand}"); return (false, blocking); }
    foreach (var S in AllCoalitions(g.N, includeEmpty: false))
    {
        if (S.Count == g.N) continue;   // grand coalition = efficacite deja teste
        double xs = S.Sum(i => x[i]);
        if (xs + 1e-9 < g.V(S))
            blocking.Add($"S={{ {string.Join(",", S.Select(i=>g.Players[i]))} }}: x(S)={xs:F4} < v(S)={g.V(S)}");
    }
    return (blocking.Count == 0, blocking);
}

// Shapley est-il dans le core du glove game ?
var (inCore, blockers) = IsInCore(glove, phiGlove);
var sb = new System.Text.StringBuilder();
sb.AppendLine("Core du glove game : Shapley est-il stable ?");
sb.AppendLine($"  Shapley = ({phiGlove[0]:F4}, {phiGlove[1]:F4}, {phiGlove[2]:F4})");
sb.AppendLine($"  Dans le core : {inCore}");
if (!inCore)
{
    sb.AppendLine("  Coalitions bloquantes :");
    foreach (var b in blockers) sb.AppendLine($"    - {b}");
}
sb.AppendLine();
// Le core du glove game : x1+x2+x3 = 1, x1+x2>=1, x1+x3>=1, x2+x3>=0, x1,x2,x3>=0.
// => x1 >= 1 - x2 et x1 >= 1 - x3, avec x2+x3 <= 1 - x1. Si x1 < 1, on a x2+x3 > 0 ok.
// Core non vide : tout x avec x1 dans [..], par ex. x=(1,0,0) : x1+x2=1>=1, x1+x3=1>=1, x2+x3=0>=0. STABLE.
sb.AppendLine("Candidat extremital x = (1, 0, 0) (L1 prend tout) :");
var extreme = new double[]{ 1.0, 0.0, 0.0 };
var (inCore2, blockers2) = IsInCore(glove, extreme);
sb.AppendLine($"  Dans le core : {inCore2}  (coalitions bloquantes : {blockers2.Count})");
sb.AppendLine();
sb.AppendLine("Lecon : le core du glove game est NON VIDE. L1 peut capturer toute la valeur");
sb.AppendLine("(x=(1,0,0) est stable : R2 et R3 seuls valent 0, donc ne peuvent bloquer).");
sb.AppendLine("C'est l'extreme : le detenteur du facteur rare peut tout extraire.");
Show(sb.ToString());


Core du glove game : Shapley est-il stable ?
  Shapley = (0,6667, 0,1667, 0,1667)
  Dans le core : False
  Coalitions bloquantes :
    - S={ L1,R2 }: x(S)=0,8333 < v(S)=1
    - S={ L1,R3 }: x(S)=0,8333 < v(S)=1

Candidat extremital x = (1, 0, 0) (L1 prend tout) :
  Dans le core : True  (coalitions bloquantes : 0)

Lecon : le core du glove game est NON VIDE. L1 peut capturer toute la valeur
(x=(1,0,0) est stable : R2 et R3 seuls valent 0, donc ne peuvent bloquer).
C'est l'extreme : le detenteur du facteur rare peut tout extraire.


### Interprétation : core du glove game

Le core du glove game est **non vide** et contient les imputations où `x_{L1} ≥ 1` contraint — en fait `x = (1, 0, 0)` est stable : L1 prend tout, et R2/R3 seuls valent 0 donc ne peuvent bloquer. Le Shapley `(2/3, 1/6, 1/6)` **n'est pas** dans le core (la coalition {L1, R2} vaut 1 mais ne reçoit que 2/3 + 1/6 = 5/6 < 1 → bloque). C'est un exemple où **Shapley ∉ core** car le jeu n'est **pas convexe**.


## 5. Jeux convexes : Shapley ∈ core

### Théorème (Shapley 1971)

Si le jeu est **convexe** (marginales croissantes : `v(S ∪ {i}) − v(S)` croît avec `S ⊆ T`), alors :
1. Le **core est non vide**.
2. La **valeur de Shapley est dans le core** (Shapley est une répartition stable).

C'est le cas « idéal » où équité (Shapley) et stabilité (core) coïncident.

### Exemple : le jeu d'aéroport

Coût de construction d'une piste dont chaque joueur `i` a besoin d'une longueur `ℓ_i`. Le coût d'une coalition `S` = `max_{i∈S} ℓ_i` (la plus grande longueur requise). Ce jeu de coût est **concave** (dual du convexe) : les économies d'échelle sont décroissantes. La répartition Shapley des coûts est dans le core.


In [7]:
// Airport game (jeu de cout) : c(S) = max longueur requise par S. Concave => Shapley des couts dans le core.
// On modelise comme un jeu (v = -cout, convexe) ou directement sur les couts.
// Longueurs requises : 3 avions (petit=1000, moyen=3000, gros=6000).
double[] lengths = { 1000.0, 3000.0, 6000.0 };
string[] planeNames = { "Petit", "Moyen", "Gros" };
int nn = 3;
// cout d'une coalition = max longueur
double Cost(List<int> S) => S.Count == 0 ? 0.0 : S.Max(i => lengths[i]);

// Shapley des couts : contribution marginale de cout.
double[] shapleyCost = new double[nn];
long cnt = 0;
foreach (var perm in Permutations(nn))
{
    var before = new List<int>();
    foreach (int i in perm)
    {
        var withI = new List<int>(before) { i };
        shapleyCost[i] += Cost(withI) - Cost(before);
        before = withI;
    }
    cnt++;
}
for (int i = 0; i < nn; i++) shapleyCost[i] /= cnt;

var sb = new System.Text.StringBuilder();
sb.AppendLine("Airport game (jeu de cout) : longueurs [1000, 3000, 6000]");
sb.AppendLine($"  Cout total (grand coalition) = max = {Cost(new List<int>{0,1,2})}");
sb.AppendLine("  Repartition Shapley des couts :");
for (int i = 0; i < nn; i++) sb.AppendLine($"    {planeNames[i]} (L={lengths[i]:F0}) paye {shapleyCost[i]:F2}");
sb.AppendLine($"    Somme = {shapleyCost.Sum():F2} (doit egaler cout total)");
sb.AppendLine();
sb.AppendLine("Lecture : le petit avion paye seulement le troncon 0-1000 (qu'il partage a parts");
sb.AppendLine("egales au debut), le moyen paye le supplement 1000-3000, le gros le 3000-6000.");
sb.AppendLine("C'est la tarification Shapley : chacun paye son increment marginal moyen.");
sb.AppendLine("Le jeu de cout etant concave, cette repartition est dans le core (stable).");
Show(sb.ToString());


Airport game (jeu de cout) : longueurs [1000, 3000, 6000]
  Cout total (grand coalition) = max = 6000
  Repartition Shapley des couts :
    Petit (L=1000) paye 333,33
    Moyen (L=3000) paye 1333,33
    Gros (L=6000) paye 4333,33
    Somme = 6000,00 (doit egaler cout total)

Lecture : le petit avion paye seulement le troncon 0-1000 (qu'il partage a parts
egales au debut), le moyen paye le supplement 1000-3000, le gros le 3000-6000.
C'est la tarification Shapley : chacun paye son increment marginal moyen.
Le jeu de cout etant concave, cette repartition est dans le core (stable).


### Interprétation : tarification Shapley de l'aéroport

La valeur de Shapley répartit le coût **par tranches d'usage marginal** : le petit avion (1000m) partage la première tranche avec tous, le gros (6000m) paie seul la dernière. C'est la base de la **tarification équitable des coûts communs** (aéroports, réseaux, projets jointifs). Le caractère concave du jeu de coût garantit la stabilité (personne ne paie plus que sa longueur seule → pas de blocage).


## Synthèse

| Concept | Formule / Test | Rôle |
|---------|----------------|------|
| **Fonction caractéristique** `v(S)` | `2^N → ℝ` | valeur d'une coalition seule |
| **Superadditivité** | `v(S∪T) ≥ v(S)+v(T)` si `S∩T=∅` | fusioner paie |
| **Convexité** | marginales `v(S∪{i})−v(S)` croissantes en `S` | stabilité garantie |
| **Shapley** `φ_i` | moyenne des marginales sur `n!` permutations | répartition équitable (4 axiomes) |
| **Banzhaf** `β_i` | nombre de coalitions swing | pouvoir de vote |
| **Core** | `Σx_i=v(N)` et `Σ_{i∈S}x_i≥v(S)` ∀S | répartitions non bloquables |
| **Convexe ⟹ Shapley ∈ core** | théorème de Shapley 1971 | équité + stabilité conjointes |


In [8]:
var sb = new System.Text.StringBuilder();
sb.AppendLine("Synthese : glove game (non convexe) vs airport game (concave)");
sb.AppendLine(new string('-', 58));
sb.AppendLine($"{"Jeu",-16}{"Shapley in core",-18}{"Convexe",-10}{"Core",-14}");
sb.AppendLine(new string('-', 58));
sb.AppendLine($"{"Glove game",-16}{"NON",-18}{"non",-10}{"non vide",-14}");
sb.AppendLine($"{"Airport (cout)",-16}{"OUI (stable)",-18}{"concave",-10}{"non vide",-14}");
sb.AppendLine(new string('-', 58));
sb.AppendLine();
sb.AppendLine("Lecon generale :");
sb.AppendLine("- jeux CONVEXES : Shapley = repartition equitable ET stable (dans le core).");
sb.AppendLine("- jeux non convexes : Shapley peut etre bloquee par une coalition ;");
sb.AppendLine("  le core peut etre vide ou contenir des imputations extremes.");
Show(sb.ToString());


Synthese : glove game (non convexe) vs airport game (concave)
----------------------------------------------------------
Jeu             Shapley in core   Convexe   Core          
----------------------------------------------------------
Glove game      NON               non       non vide      
Airport (cout)  OUI (stable)      concave   non vide      
----------------------------------------------------------

Lecon generale :
- jeux CONVEXES : Shapley = repartition equitable ET stable (dans le core).
- jeux non convexes : Shapley peut etre bloquee par une coalition ;
  le core peut etre vide ou contenir des imputations extremes.


### Interprétation du tableau comparatif

La convexité (ou concavité d'un jeu de coût) est la condition clé : elle **garantit** que la répartition équitable (Shapley) est aussi **stable** (dans le core). Hors convexité, équité et stabilité divergent — c'est le cœur de la difficulté des jeux coopératifs non convexes (négociation, externalités).


## 7. Exercices

> Stubs à compléter (jamais `throw` : le notebook s'exécute end-to-end, règle C.1).

### Exercice 1 — Jeu de vote pondéré (Conseil de l'UE)

Construire un jeu de vote pondéré représentant le Conseil de l'UE (poids des grands pays) et calculer les indices de Banzhaf. Qui a le plus de pouvoir relatif ?


In [9]:
// Exercice 1 — Conseil UE (a completer).
#nullable enable
// TODO etudiant : definir les poids des pays (FR, DE, IT, ES = 29 chacun, etc.) et le quota (62%).
// Etape 1 : WeightedVotingGame + Banzhaf. Etape 2 : comparer beta_i normalise au poids.
Console.WriteLine("Exercice 1 (Conseil UE) a completer.");


Exercice 1 (Conseil UE) a completer.


### Exercice 2 — Bankruptcy game

Trois créanciers réclament 100, 200, 300 ; l'actif disponible est 350. Modéliser comme jeu coopératif (`v(S) = max(0, actif − somme des réclamations hors S)`) et calculer Shapley. Comparer à la règle proportionnelle.


In [10]:
// Exercice 2 — Bankruptcy game (a completer).
#nullable enable
// TODO etudiant : v(S) = max(0, E - sum_{i notin S} claim_i). Shapley vs proportionnel.
Console.WriteLine("Exercice 2 (Bankruptcy) a completer.");


Exercice 2 (Bankruptcy) a completer.


### Exercice 3 — Core vide

Construire un jeu à 3 joueurs dont le core est **vide** (ex: jeu non équilibré) et le vérifier (aucune imputation stable).


In [11]:
// Exercice 3 — Core vide (a completer).
#nullable enable
// TODO etudiant : definir v tq aucune imputation x (sum=v(N), x(S)>=v(S) forall S) n'existe.
// Indice : jeu a majority avec 3 joueurs, v(S)=1 si |S|>=2, v(N)=1 => core vide.
Console.WriteLine("Exercice 3 (Core vide) a completer.");


Exercice 3 (Core vide) a completer.


### Exercice 4 — Nucleolus

Le **nucleolus** est la répartition qui minimise lexicographiquement le vecteur des excès (mécontentement max des coalitions). L'implémenter (avancé) et comparer à Shapley sur le glove game.


In [12]:
// Exercice 4 — Nucleolus (a completer).
#nullable enable
// TODO etudiant : trier les excès e_S(x) = v(S) - x(S), minimiser le max lexicographiquement.
Console.WriteLine("Exercice 4 (Nucleolus) a completer.");


Exercice 4 (Nucleolus) a completer.


### Exercice 5 — Convexité et core

Montrer (par énumération) que pour un jeu convexe à 3 joueurs, la valeur de Shapley satisfait toutes les inégalités du core.


In [13]:
// Exercice 5 — Convexite => Shapley in core (a completer).
#nullable enable
// TODO etudiant : construire un jeu convexe (v convex), calculer Shapley, verifier IsInCore.
Console.WriteLine("Exercice 5 (Convexite) a completer.");


Exercice 5 (Convexite) a completer.


## 8. Résumé

### Points clés

- **Jeu coopératif** `(N, v)` : `v(S)` = valeur d'une coalition.
- **Shapley** : répartition équitable (contribution marginale moyenne, 4 axiomes).
- **Banzhaf** : pouvoir de swing (poids ≠ pouvoir).
- **Core** : répartitions non bloquables (efficacité + rationalité coalitionnelle).
- **Convexité** ⟹ core non vide **et** Shapley ∈ core (équité + stabilité).

### Leçon centrale

La théorie coopérative sépare **équité** (Shapley) et **stabilité** (core). Elles coïncident pour les jeux convexes ; sinon, il faut choisir — ou chercher d'autres solutions (nucleolus, valeur de Nash).

### Prochaine étape

- [GameTheory-16 (Choix social / Mechanism Design)](GameTheory-16-MechanismDesign-Csharp.ipynb) : agrégation des préférences et conception d'enchères.

---

**Twin C# .NET Interactive** du notebook Python `GameTheory-15-CooperativeGames.ipynb`. Implémentation from-scratch (BCL .NET 9, 0 NuGet) — marathon [#4956](https://github.com/jsboige/CoursIA/issues/4956), registre SOTA [#3801](https://github.com/jsboige/CoursIA/issues/3801).
